In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D1_05 EDA 체크리스트

> **오늘 질문:** 이 데이터를 믿어도 될까? 고장은 어디서 많이 나나?

4교시에 만든 분석 테이블을 **EDA 체크리스트 10단계** 순서대로 점검합니다.

| # | 확인 항목 | # | 확인 항목 |
|---|---|---|---|
| 1 | 크기 `shape` | 6 | 기술통계 `describe()` |
| 2 | 타입 `dtypes` | 7 | 범주 빈도 `value_counts()` |
| 3 | 결측 `isna().sum()` | 8 | 타깃 분포: 고장 비율 |
| 4 | 중복 `duplicated()` | 9 | 그룹 비교: 설비·등급별 |
| 5 | 범위: 가능한 값인가 | 10 | 시간 추이 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 데이터 사전

> **1행 = 생산 1건**(제품 1개)의 센서 측정값과 품질 결과입니다.
> 4교시에서 합친 네 파일의 컬럼이 모두 들어 있습니다.

| 컬럼 | 의미 | 단위·값 |
|---|---|---|
| `record_id` | 생산 기록 번호 | 정수 |
| `timestamp` | 측정 일시 | 날짜·시간 |
| `equip_id` | 설비 ID | `MC-01` ~ `MC-10` |
| `product_id` | 제품 ID | 문자 (앞 글자가 등급) |
| `grade` | 제품 등급 | `L`(저) / `M`(중) / `H`(고) |
| `air_temp` | 주변 온도 | ℃ |
| `process_temp` | 공정 온도 | ℃ |
| `rpm` | 회전수 | rpm |
| `torque` | 토크 | Nm |
| `tool_wear` | 공구 마모 시간 | 분 |
| `vibration` | 진동 | mm/s |
| `failure` | 고장 여부 (**타깃**) | 0 = 정상, 1 = 고장 |
| `twf` | 공구 마모 고장 (Tool Wear Failure) | 0 / 1 |
| `hdf` | 방열 불량 고장 (Heat Dissipation Failure) | 0 / 1 |
| `pwf` | 전력 이상 고장 (Power Failure) | 0 / 1 |
| `osf` | 과부하 고장 (Overstrain Failure) | 0 / 1 |
| `rnf` | 무작위 고장 (Random Failure) | 0 / 1 |
| `line` | 설비가 속한 라인 | `L1`, `L2` |
| `maker` | 설비 제조사 | `A사`, `B사`, `C사` |
| `install_year` | 설비 설치 연도 | 연도 |

<div style="clear: both; height: 20px;"></div>

- `line`, `maker`, `install_year`는 설비 단위 정보라 같은 설비의 모든 행에 같은 값이 들어갑니다.

## 0. 준비 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

# 4교시 결과물과 같은 분석 테이블
df = pd.read_parquet("../../data/analysis_table.parquet")

## 1~2. 크기와 타입 ✍️

In [ ]:
# 힌트: df.shape, df.dtypes

## 3. 결측 ✍️

결측이 몇 개인지보다 **어디에 몰려 있는지**가 중요합니다.

In [ ]:
# 힌트: df.isna().sum()

결측이 있는 행만 골라 특정 설비나 시간에 몰려 있는지 봅니다.

In [ ]:
# 힌트: df[df["컬럼"].isna()]

특정 설비에 몰려 있지 않으면, 이번에는 **중앙값으로 채웁니다.**

In [ ]:
# 힌트: df["컬럼"].fillna(값)

## 4. 중복 ✍️

완전히 같은 행이 몇 개인지 세고, 지운 뒤 행 수를 다시 확인합니다.

In [ ]:
# 힌트: df.duplicated().sum(), df.drop_duplicates()

## 5~6. 범위와 기술통계 ✍️

최솟값·최댓값이 물리적으로 가능한 값인지 봅니다. (온도는 ℃, 회전수는 rpm, 토크는 Nm, 공구 마모는 분, 진동은 mm/s)

In [ ]:
# 힌트: df.describe()

**Q1.** 물리적으로 이상해 보이는 값이 있나요? 평균과 중앙값(50%)이 크게 다른 컬럼은?

> 답:


## 7~8. 범주 빈도와 고장 비율 ✍️

In [ ]:
# 힌트: df["컬럼"].value_counts(), df["컬럼"].mean()

## 9. 그룹 비교 ✍️

불량률은 **분모(생산 건수)와 함께** 봅니다. `agg`로 건수·고장 수·고장률을 한 번에 구합니다.

In [ ]:
# 힌트: df.groupby(["컬럼1", "컬럼2"])["failure"].agg(["count", "sum", "mean"])

## 10. 시간 추이 ✍️

`resample("W", on="timestamp")`는 일시를 주 단위로 묶습니다.

In [ ]:
# 힌트: df.resample("W", on="timestamp")["컬럼"].mean().plot()

고장률이 갑자기 튄 주가 있습니다. 그 주(2월 10~16일)의 고장 유형을 세어 봅니다. 📋

In [ ]:
week = df[(df["timestamp"] >= "2025-02-10") & (df["timestamp"] < "2025-02-17")]
print(f"이 주의 고장: {week['failure'].sum()}건 / 전체 기간 고장: {df['failure'].sum()}건")
week[["twf", "hdf", "pwf", "osf", "rnf"]].sum()

**Q2.** 고장률이 튄 주에는 어떤 고장 유형이 몰려 있나요? 현장이라면 무엇을 먼저 확인하겠습니까?

> 답:


## 정리 💬

**Q3.** 고장률이 눈에 띄게 높은 설비는 어디인가요? 설비 정보에서 공통점을 찾아보세요.

> 답:


**Q4.** 결측과 중복을 처리하지 않고 설비별 고장률을 냈다면 무엇이 달라졌을까요?

> 답:
